In [2]:
import os
import sys
import importlib

from typing import Any, Dict, Counter, List


sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.datatypes as datatypes

importlib.reload(datatypes)

UserTransition = datatypes.UserTransition
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey


In [ ]:
class DrlPolicy(CachePolicy):
    def __init__(self, cfg: Any = None):
        self.cfg = cfg
        self.cur_size = 0

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def get(self, key: CacheKey) -> Any:
        return self.cache.get(key, None)

    def put(self, key: int, value: Any, size: int) -> list:
        """
        key   -> slot index
        value -> (video_id, tiles)
        size  -> fixed as 1 slot
        """
        evicted = []
        slot = key
        new_video, _ = value

        if new_video in self.video_idx:
            # old_video_slot = self.video_idx.index(new_video)
            # if old_video_slot != slot:
            #     # Swap video positions
            #     self.video_idx[slot], self.video_idx[old_video_slot] = (
            #         self.video_idx[old_video_slot],
            #         self.video_idx[slot],
            #     )

            #     # Swap tile mapping accordingly
            #     self.tile_idx[slot], self.tile_idx[old_video_slot] = (
            #         self.tile_idx[old_video_slot],
            #         self.tile_idx[slot],
            #     )
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return evicted

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = [-1] * self.cfg.viewport

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return evicted

    def contains(self, key: CacheKey) -> bool:
        return key in self.video_idx

    def remove(self, key: CacheKey) -> bool:
        if key in self.video_idx:
            idx = self.video_idx.index(key)
            self.video_idx[idx] = -1
            self.tile_idx[idx] = [-1] * self.cfg.viewport
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return True
        return False

    def clear(self) -> None:
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]
        self.cur_size = 0

    def keys(self):
        return self.video_idx
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def update_size(self):
        self.cur_size = sum(1 for v in self.video_idx if v != -1)

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [ ]:
class MMSPPolicy(CachePolicy):
    def __init__(self, cfg: Any = None):
        self.cfg = cfg
        self.cur_size = 0
        self.capacity = self.cfg.cache_size * (self.cfg.viewport + 1)

        self.cache = [(-1,-1) for _ in range(self.capacity)]

    def get(self, key: CacheKey) -> Any:
        return self.cache.get(key, None)

    def _put_base_layer(self, key: int, value: Any) -> None:

        slot = key

        old_video, old_tile = self.cache[slot]
        self.cache[slot] = value

        if old_tile != -1:
            return
        
        for i, (cached_video, _) in enumerate(self.cache):
            if cached_video == old_video:
                self.cache[i] = (-1, -1)

        return

    def _put_enh_layer(self, key: int, value: Any) -> None:
        slot = key

        old_video, old_tile = self.cache[slot]
        self.cache[slot] = value

        if old_tile != -1:
            return
        
        for i, (cached_video, _) in enumerate(self.cache):
            if cached_video == old_video:
                self.cache[i] = (-1, -1)

        return

    def put(self, key: int, value: Any, size: int) -> list:
        """
        key   -> slot index
        value -> (video_id, tiles)
        size  -> fixed as 1 slot
        """
        evicted = []
        slot = key
        _, new_tile = value

        if value in self.cache:
            return evicted

        if new_tile == -1:
            self._put_base_layer(slot, value)
        else:
            self._put_enh_layer(slot, value)

        return evicted

    def contains(self, key: CacheKey) -> bool:
        return key in self.video_idx

    def remove(self, key: CacheKey) -> bool:
        if key in self.video_idx:
            idx = self.video_idx.index(key)
            self.video_idx[idx] = -1
            self.tile_idx[idx] = [-1] * self.cfg.viewport
            self.cur_size = sum(1 for v in self.video_idx if v != -1)
            return True
        
        return False

    def clear(self) -> None:
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]
        self.cur_size = 0

    def keys(self):
        return self.video_idx
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def update_size(self):
        self.cur_size = sum(1 for v in self.video_idx if v != -1)

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }